In [1]:
## Libraries, modules, etc.
import os
import pandas as pd
import numpy as np
import soundfile as sf
import cv2
import time
import multiprocessing
from functools import partial
from skimage.feature import peak_local_max
from scipy import signal
from seq_utils import (
    select_win, spec_bandpass, meas_time_freq, signals2list, blank_spectrogram, remove_short_detections, get_tone_segments,
    find_tone_signals, translate_tones, sig_length, compute_length_metric, validate_list
)

start_time = time.time()

## Input directory with pre-processed sequence audio files
directory = "./A_signal_sequences/"

## Audio and FFT Parameters
fs = 48000 # Sampling frequency
win_type = "Taylor"
win_size = 2400  # Window size, 1000 to 4000 are ideal
overlap = 80 # Overlap %

## Initialize FFT
win = select_win(win_type, win_size) # FFT window
hop_size = int(win_size*(1-overlap/100)) # Step/hop size
FFT_sample_T = win_size/fs*(1-overlap/100) # FFT samples duration in s (this is not actual resolution)
SFT = signal.ShortTimeFFT(win, hop=hop_size, fs=fs) # Initialize ShortTimeFFT

# Signal samples to be used in template matching
signal_templates = {
    'X': np.genfromtxt('signal_samples/EOL_avg2_48k-2400-80-Taylor.csv', delimiter=',').astype(np.float32),
    'G': np.genfromtxt('signal_samples/G_avg2_48k-2400-80-Taylor.csv', delimiter=',').astype(np.float32),
    'F': np.genfromtxt('signal_samples/F_avg2_48k-2400-80-Taylor.csv', delimiter=',').astype(np.float32),
    'E': np.genfromtxt('signal_samples/E_avg2_48k-2400-80-Taylor.csv', delimiter=',').astype(np.float32),
    'D': np.genfromtxt('signal_samples/D_avg2_48k-2400-80-Taylor.csv', delimiter=',').astype(np.float32),
}

## Big function with all the signal processing steps
def process_sequence(rec_filename, directory, signal_templates):
    file_path = os.path.join(directory, rec_filename)
    seq_name = int(rec_filename.split("_")[0])
    df = pd.DataFrame(columns=["Sequence", "Signal Type", "Time (s)", "Frequency (Hz)", "Correlation Value"])
    signals_detected = []

    try:
        ## Read sequence audio recording
        audio_rec, fs = sf.read(file_path)
        data = audio_rec[:, 0] + audio_rec[:, 1] # L+R channel summing

        ## Generate spectrogram
        Sx2 = SFT.spectrogram(data)
        Sx2[Sx2 == 0] = 1e-10
        Sx2_dB = 10 * np.log10(Sx2)
        t_min, t_max, f_min, f_max = SFT.extent(len(audio_rec), center_bins=True)
        f_bins, t_bins = Sx2.shape
        time_axis = np.linspace(t_min, t_max, t_bins)
        freq_axis = np.linspace(f_min, f_max, f_bins)
    
        # Crop spectrogram to avoid high frequency aliased signals, blank first 9.5 s and last 9.7 s, they shouldn't contain any signal
        recording_crop = spec_bandpass(Sx2_dB, f_max, f_bins, 0, 14000).astype(np.float32)
        recording_crop[:, 0:int(9.5/FFT_sample_T)] = -100
        recording_crop[:, -int(9.7/FFT_sample_T):] = -100        
        
        ## Detect EOL signal
        # Time corrections
        t_start_corr = 16.18-3.90 # Correction to signal start time (s)
        t_end_corr = 0.1 # Correction to signal end time (s)        
        # Signal detection
        signal = signal_templates['X'] # Load signal
        correlation_opencv = cv2.matchTemplate(recording_crop, signal, cv2.TM_CCOEFF_NORMED) # Signal correlation search based on OpenCV match template
        local_maxima = peak_local_max(correlation_opencv, min_distance=10, threshold_abs=0.52) # Find local maxima locations
        detected_times, detected_freqs = meas_time_freq(local_maxima, time_axis, freq_axis) # Convert the local maxima indices to time and frequency values
        correl_values = correlation_opencv[local_maxima[:, 0], local_maxima[:, 1]] # Extract correlation values
        signals_detected = signals2list("X", signals_detected, detected_times-t_start_corr, detected_freqs, correl_values) # Add to list of detected signals        
        # Blank spectrogram
        recording_crop_blank_EOL = blank_spectrogram(recording_crop, local_maxima, t_start_corr, t_end_corr, signal, t_bins, FFT_sample_T)        
        
        ## Detect G signal
        # Time corrections
        t_start_corr = 0 # Correction to signal start time (s)
        t_end_corr = 0 # Correction to signal end time (s)        
        # Signal detection
        signal = signal_templates['G'] # Load signal
        correlation_opencv = cv2.matchTemplate(recording_crop_blank_EOL, signal, cv2.TM_CCOEFF_NORMED) # Signal correlation search based on OpenCV match template
        local_maxima = peak_local_max(correlation_opencv, min_distance=10, threshold_abs=0.45) # Find local maxima locations
        detected_times, detected_freqs = meas_time_freq(local_maxima, time_axis, freq_axis) # Convert the local maxima indices to time and frequency values
        correl_values = correlation_opencv[local_maxima[:, 0], local_maxima[:, 1]] # Extract correlation values
        signals_detected = signals2list("G", signals_detected, detected_times-t_start_corr, detected_freqs, correl_values) # Add to list of detected signals        
        # Blank spectrogram
        recording_crop_blank_G = blank_spectrogram(recording_crop_blank_EOL, local_maxima, t_start_corr, t_end_corr, signal, t_bins, FFT_sample_T)        
        
        ## Detect F signal
        # Time corrections
        t_start_corr = 0 # Correction to signal start time (s)
        t_end_corr = 0 # Correction to signal end time (s)        
        # Signal detection
        signal = signal_templates['F'] # Load signal
        correlation_opencv = cv2.matchTemplate(recording_crop_blank_G, signal, cv2.TM_CCOEFF_NORMED) # Signal correlation search based on OpenCV match template
        local_maxima = peak_local_max(correlation_opencv, min_distance=10, threshold_abs=0.42) # Find local maxima locations
        detected_times, detected_freqs = meas_time_freq(local_maxima, time_axis, freq_axis) # Convert the local maxima indices to time and frequency values
        correl_values = correlation_opencv[local_maxima[:, 0], local_maxima[:, 1]] # Extract correlation values
        signals_detected = signals2list("F", signals_detected, detected_times-t_start_corr, detected_freqs, correl_values) # Add to list of detected signals
        # Blank spectrogram
        recording_crop_blank_F = blank_spectrogram(recording_crop_blank_G, local_maxima, t_start_corr, t_end_corr, signal, t_bins, FFT_sample_T)        
        
        ## Detect E signal
        # Time corrections
        t_start_corr = 1.27-0.89 # Correction to signal start time (s)
        t_end_corr = 0 # Correction to signal end time (s)        
        # Signal detection
        signal = signal_templates['E'] # Load signal
        correlation_opencv = cv2.matchTemplate(recording_crop_blank_F, signal, cv2.TM_CCOEFF_NORMED) # Signal correlation search based on OpenCV match template
        local_maxima = peak_local_max(correlation_opencv, min_distance=10, threshold_abs=0.48) # Find local maxima locations
        detected_times, detected_freqs = meas_time_freq(local_maxima, time_axis, freq_axis) # Convert the local maxima indices to time and frequency values
        correl_values = correlation_opencv[local_maxima[:, 0], local_maxima[:, 1]] # Extract correlation values
        signals_detected = signals2list("E", signals_detected, detected_times-t_start_corr, detected_freqs, correl_values) # Add to list of detected signals        
        # Blank spectrogram
        recording_crop_blank_E = blank_spectrogram(recording_crop_blank_F, local_maxima, t_start_corr, t_end_corr, signal, t_bins, FFT_sample_T)        
        
        ## Detect D signal
        # Time corrections
        t_start_corr = 1.27-0.89 # Correction to signal start time (s)
        t_end_corr = 0 # Correction to signal end time (s)        
        # Signal detection
        signal = signal_templates['D'] # Load signal
        correlation_opencv = cv2.matchTemplate(recording_crop_blank_E, signal, cv2.TM_CCOEFF_NORMED) # Signal correlation search based on OpenCV match template
        local_maxima = peak_local_max(correlation_opencv, min_distance=10, threshold_abs=0.45) # Find local maxima locations
        detected_times, detected_freqs = meas_time_freq(local_maxima, time_axis, freq_axis) # Convert the local maxima indices to time and frequency values
        correl_values = correlation_opencv[local_maxima[:, 0], local_maxima[:, 1]] # Extract correlation values
        signals_detected = signals2list("D", signals_detected, detected_times-t_start_corr, detected_freqs, correl_values) # Add to list of detected signals        
        # Blank spectrogram
        t_start_corr = 0 # Correction to signal start time (s)
        t_end_corr = 1.61-0.31 # Correction to signal end time (s)
        recording_crop_blank_D = blank_spectrogram(recording_crop_blank_E, local_maxima, t_start_corr, t_end_corr, signal, t_bins, FFT_sample_T)        
    
        ## Preliminary tone detection
        # Measure background noise level
        noise = spec_bandpass(Sx2_dB, f_max, f_bins, 6300, 6700) # Extract noise band
        noise_mean = np.mean(noise) # Noise mean level
        noise_std = np.std(noise) # Noise standard deviation
                
        # Measure signal level in strong "tone" band
        sig_bandpass = spec_bandpass(recording_crop_blank_D, f_max, f_bins, 7350, 7650) # Apply bandpass
        sig_level = np.max(sig_bandpass, axis=0) # Peak values for each time sample
        peak_mean = np.mean(sig_level[sig_level > (np.max(sig_level) - 6)]) # Calculate mean of peak values
        peak_std = np.std(sig_level[sig_level > (np.max(sig_level) - 6)]) # Calculate std of peak values
        
        # Thresholding
        sig_max = np.max(sig_level)
        threshold = peak_mean - peak_std*2 - noise_std*0.75 # Threshold level to detect a signal        
        above_threshold = (sig_level > threshold).astype(int) # Array of detections (1) and non-detections (0)
        min_ones = int(0.26/FFT_sample_T) # 0.26 seconds, = 0.67 * 0.38 s (2/3 of the shortest portion of signal C)
        sig_detected = remove_short_detections(above_threshold, min_ones)
                
        ## Blanking of noise sections based on previous detections, then bandpass to low frequency strong tone.
        # Cropping based on sig_detected, applied to the last processed spectrogram (after D signal detection)
        recording_crop_blank_D[:, np.where(sig_detected == 0)[0]] = -100        
        # First bandpass filter (wide)
        f_low_1 = 6500
        f_high_1 = 7200
        recording_crop_blank_tones_bp1 = spec_bandpass(recording_crop_blank_D, f_max, f_bins, f_low_1, f_high_1)
                
        ## Find local maxima for each tone (frequency sample), 1st pass
        tones_local_max = np.argmax(recording_crop_blank_tones_bp1, axis=0) # Local maxima, this is often noisy
        det_idx, tone_segm_boundaries = get_tone_segments(sig_detected) # Find indices of tone segments        
        # Calculate the median of tones_local_max for each tone segment
        medians = np.zeros(len(tone_segm_boundaries) - 1)
        for i in range(len(medians)):
            segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
            medians[i] = np.median(tones_local_max[segment])        
        # Initialize an array for tones_local_max_med, add the corresponding median for each segment (this works as a filter)
        tones_freq_med = np.copy(tones_local_max)
        for i in range(len(medians)):
            segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
            tones_freq_med[segment] = medians[i]
        
        ## Determine min/max tone frequencies, then run second bandpass (narrow)
        margin = 50 # Frequency margin below and above tones
        f_low_2 = np.min(tones_freq_med[tones_freq_med != 0])*fs/win_size + f_low_1 - margin
        f_high_2 = np.max(tones_freq_med)*fs/win_size + f_low_1 + margin
        recording_crop_blank_tones_bp2 = spec_bandpass(recording_crop_blank_D, f_max, f_bins, f_low_2, f_high_2)
                
        ## Find local maxima for each tone (frequency sample), 2nd pass
        tones_local_max = np.argmax(recording_crop_blank_tones_bp2, axis=0) # Local maxima, this is often noisy
        det_idx, tone_segm_boundaries = get_tone_segments(sig_detected) # Find indices of tone segments        
        # Calculate the median of tones_local_max for each tone segment
        medians = np.zeros(len(tone_segm_boundaries) - 1)
        for i in range(len(medians)):
            segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
            medians[i] = np.median(tones_local_max[segment])        
        # Initialize an array for tones_local_max_med, add the corresponding median for each segment (this works as a filter)
        tones_freq_med = np.copy(tones_local_max)
        for i in range(len(medians)):
            segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
            tones_freq_med[segment] = medians[i]
                
        ## Detect tones, error check, length metric
        # Find tone signals (as sequences of values above 0)
        tone_signals = find_tone_signals(sig_detected, tones_freq_med)
    
        # Detect which tones occur, with error checking
        try:
            tones_list = translate_tones(tone_signals, FFT_sample_T, seq_name)
        except ValueError as e:
            print(f"Seq {seq_name} Error: {e}")
            print(f"Seq {seq_name} Error(s) found, sequence not transcribed.")
            return None
        
        # Convert to time and frequency values
        tone_type = np.array(tones_list)[:, 3]
        tone_times = np.array([row[0] for row in tones_list])
        tone_freqs = np.array([row[4] for row in tones_list])
        detected_times = np.linspace(t_min, t_max, t_bins)[tone_times] # Time values from indices
        detected_freqs = np.linspace(f_min, f_max, f_bins)[tone_freqs] + f_low_2  # Frequency values from indices, corrected for previous bandpass
        
        # Calculate length metric
        length_metric = compute_length_metric(tones_list, tone_type, sig_length, FFT_sample_T)
        
        # Add to list of detected signals
        for i in range(len(detected_times)):
            signals_detected.append([
                tone_type[i], # Signal type
                round(detected_times[i], 2), # Time rounded to 2 decimal places
                int(detected_freqs[i]), # Frequency as an integer
                round(length_metric[i], 2) # Length metric as an integer
            ])
                    
        ## Final signal sorting and validation
        signals_detected = sorted(signals_detected, key=lambda x: x[1])
        if validate_list(signals_detected, seq_name):
            for row in signals_detected:
                df.loc[len(df)] = [seq_name] + row

    except Exception as e:
        print(f"Error processing {rec_filename}: {e}")
        return None

    return df

## Find all files and run sequence trascription in parallel
all_files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
with multiprocessing.Pool(processes=24) as pool:
    process_fn = partial(process_sequence, directory=directory, signal_templates=signal_templates)
    results = pool.map(process_fn, all_files)

df = pd.concat(results, ignore_index=True)

print(f"Done! Processed {len(all_files)} sequences, detected {len(df)} signals.")
print(f"Execution time: {time.time() - start_time:.2f} seconds")

Done! Processed 308 sequences, detected 7559 signals.
Execution time: 62.32 seconds


In [2]:
# Round Time (s) and Correlation Value to 2 decimal places
df['Time (s)'] = df['Time (s)'].round(2)
df['Correlation Value'] = df['Correlation Value'].round(2)

# Ensure Sequence and Frequency (Hz) are integers
df['Sequence'] = df['Sequence'].astype(int)
df['Frequency (Hz)'] = df['Frequency (Hz)'].astype(int)

# Export to CSV
df.to_csv('transcribed_sequences/A_transcribed_sequences.csv', index=False)